# S2D emotion recognition pipeline — ElderReact & Alzheimer patient videos

This notebook runs the whole S2D track step by step, **in Google Colab or locally** on your PC. Every step calls one of the scripts in `s2d/` (the same ones you can run from a terminal), shows the results here, and writes them to the results folder.

| Step | What it does | Needs a GPU | Time (rough) |
|---|---|---|---|
| 0 | Setup: Colab/Drive or local, code, packages | – | 2–3 min |
| 1 | Settings (paths, training options, window preset) | – | – |
| 2 | Train / validation split (pooled train+dev, like the MAE-DFER track) | – | seconds |
| 3 | Baseline: MAFW model on ElderReact *before* fine-tuning | yes | ~2 min |
| 4 | Fine-tune S2D on ElderReact | yes | ~25–60 min |
| 5 | Evaluate the fine-tuned model (validation → thresholds → test) | yes | ~2 min |
| 6 | Compare models with 95 % confidence intervals (optionally incl. MAE-DFER) | – | seconds |
| 7 | t-SNE plot before vs after | – | ~30 s |
| 8 | 3 annotated frames per patient video | yes | ~1 min / video |
| 9 | Emotion timeline per patient video | yes | ~5–15 min / video |
| A | *(optional)* face-crop new videos | yes | depends |
| B | *(optional)* MAFW fold-1 check of the released model | yes | ~5 min |

Steps whose output already exists are **skipped** (set `REDO = True` in the settings to force them). So after a Colab disconnect you can simply run all cells again.

## Before you start

### Option A — Google Colab (recommended for training)

**1. Put the code on GitHub.** The notebook clones the `s2d` branch of the project repository, so push your latest changes first:
```
git add s2d results/*.txt
git commit -m "S2D pipeline notebook"
git push origin s2d
```

**2. Put the data on Google Drive** in `MyDrive/emotion_s2d/` (create the folders yourself):
```
MyDrive/emotion_s2d/
├── checkpoints/
│   └── split01_checkpoint-best.pth        ← from external/S2D/ckpts/MAFW/without_oversampler/
├── data/
│   ├── ElderReact_processed.tar           ← face crops + label files (see below)
│   ├── AlzheimerPatient_Data/             ← the patient videos (Img 6705.mp4, ...)
│   └── MAFW_test_deepface.tar             ← optional, only for step B
├── splits/                                ← optional: validation list from the MAE-DFER track
└── results/                               ← created by the notebook; everything is saved here
```

The face crops are ~200,000 small images. Uploading them one by one to Drive is very slow, so pack them into **one file** first. In PowerShell, from the repo folder (`tar` is built into Windows 10/11):
```
tar -cf ElderReact_processed.tar -C data/processed ElderReact
tar -cf MAFW_test_deepface.tar -C data/processed/MAFW test_deepface      # optional
```
`ElderReact_processed.tar` then contains `ElderReact/faces_224/...` and `ElderReact/anno/*.csv`. The notebook unpacks it onto Colab's fast local disk at the start of each session (a few minutes).

**3. Open this notebook in Colab** (File → Open notebook → GitHub, or upload it), then **Runtime → Change runtime type → GPU** (T4 is enough; L4/A100 are faster), and run the cells from the top. Colab asks for permission to access your Drive once per session.

### Option B — locally on your PC
Use the `venv-s2d` environment as the notebook kernel. Once: `pip install ipykernel` inside `venv-s2d`, then in VS Code / Jupyter pick `venv-s2d` as the kernel. Open the notebook from the repo's `s2d/` folder. Paths are the usual ones in the repo (`data/processed/...`, `external/S2D/...`, `results/`).

## Step 0 — Setup

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/LukasStrandberg/Scientific-computing-project---Emotion-detection.git"
BRANCH = "s2d"
DRIVE_DIR = Path("/content/drive/MyDrive/emotion_s2d")   # Colab only

def sh(cmd):
    """Run a shell command and stop on errors."""
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = Path("/content/Scientific-computing-project---Emotion-detection")
    if REPO.exists():
        sh(f"git -C '{REPO}' pull -q")
    else:
        sh(f"git clone -q -b {BRANCH} {REPO_URL} '{REPO}'")
    if not (REPO / "external/S2D").exists():
        sh(f"git clone -q https://github.com/MSA-LMC/S2D.git '{REPO}/external/S2D'")
    # Colab already has PyTorch with CUDA; S2D needs this exact timm version
    sh("pip install -q timm==0.6.13 einops tensorboardX batch-face")
else:
    REPO = Path.cwd().parent if Path.cwd().name == "s2d" else Path.cwd()
    if not (REPO / "external/S2D").exists():
        raise SystemExit("external/S2D is missing - see s2d/README.md section 1 (git clone of the official S2D code)")

os.chdir(REPO)
import torch
print("Repo:", REPO)
print("Running in", "Colab" if IN_COLAB else "local mode", "| PyTorch", torch.__version__)
if torch.cuda.is_available():
    GPU_MEM = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({GPU_MEM:.0f} GB)")
else:
    GPU_MEM = 0
    print("WARNING: no GPU found - in Colab use Runtime -> Change runtime type -> GPU")

In [ ]:
# Helpers used by all steps
from IPython.display import Image, display, Markdown

def run(script, *args):
    """Run s2d/<script> with the kernel's Python and stream its output here."""
    cmd = [sys.executable, "-u", str(REPO / "s2d" / script)] + [str(a) for a in args]
    short = [str(a).replace(str(REPO) + os.sep, "") for a in args]   # paths relative to the repo, for readability
    print("$ python s2d/" + script, " ".join(f'"{a}"' if " " in a else a for a in short), "\n")
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if "FutureWarning" in line or "warnings.warn" in line:
            continue
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{script} failed (exit code {proc.returncode}) - see the output above")

def done(path):
    """True if a step's output exists and REDO is off (then the step is skipped)."""
    if Path(path).exists() and not REDO:
        print(f"Already done ({path}) - skipped. Set REDO = True in the settings to run it again.")
        return True
    return False

def show_text(path):
    print(Path(path).read_text())

## Step 1 — Settings
Everything you might want to change is in this cell.

In [ ]:
REDO = False                     # True: run steps again even if their output exists

# ---------- paths ----------
if IN_COLAB:
    DATA = Path("/content/data/processed")                       # unpacked from the tar files on Drive
    CKPT_MAFW = DRIVE_DIR / "checkpoints/split01_checkpoint-best.pth"
    PATIENT_VIDEOS = DRIVE_DIR / "data/AlzheimerPatient_Data"
    RESULTS = DRIVE_DIR / "results"                              # on Drive, so it survives disconnects
    TARS = {DATA / "ElderReact": DRIVE_DIR / "data/ElderReact_processed.tar",
            DATA / "MAFW/test_deepface": DRIVE_DIR / "data/MAFW_test_deepface.tar"}
else:
    DATA = REPO / "data/processed"
    CKPT_MAFW = REPO / "external/S2D/ckpts/MAFW/without_oversampler/split01_checkpoint-best.pth"
    PATIENT_VIDEOS = REPO / "data/AlzheimerPatient_Data"
    RESULTS = REPO / "results"
    TARS = {}

ER_FRAMES = DATA / "ElderReact/faces_224"
ER_ANNO = DATA / "ElderReact/anno"            # train.csv, dev.csv, test.csv
MAFW_FRAMES = DATA / "MAFW/test_deepface"     # optional (step B)
MAFW_ANNO = DATA / "MAFW/anno/set_1/test.csv"

# ---------- train / validation split (step 2) ----------
SPLIT = "pooled"         # "pooled": train+dev pooled and re-split (MAE-DFER protocol) | "official": train / dev as given
SPLIT_FILE = None        # e.g. DRIVE_DIR / "splits/val_clips.txt" - exact validation clips from the MAE-DFER run
VAL_FRAC = 0.15          # pooled split without SPLIT_FILE: share of train+dev used for validation
SPLIT_SEED = 42
SPLIT_DIR = RESULTS / (f"split_{SPLIT}" + ("" if SPLIT_FILE is None else "_file"))

# ---------- fine-tuning (step 4) ----------
RUN_NAME = f"er_ft_{SPLIT}"   # results go to RESULTS/<RUN_NAME>
EPOCHS = 30              # upper limit; early stopping usually ends sooner
PATIENCE = 8             # stop after this many epochs without improvement (0 = off)
SELECT = "val_loss"      # best epoch by "val_loss" (MAE-DFER protocol) or "auc"
LR = 5e-5                # learning rate (5e-5 worked best in our runs; 1e-3 for TRAINABLE="head")
TRAINABLE = "adapters"   # "adapters" = MCP + TMA + classifier (official S2D) | "head" = classifier only
BATCH, ACCUM = (8, 2) if GPU_MEM >= 20 else (2, 8)   # effective batch 16 either way; lower BATCH if out of memory

# ---------- patient videos (steps 8, 9) ----------
WINDOW_PRESET = "s2d"    # "s2d": 3 s windows every 1 s, smoothed | "maedfer": 1 s every 0.8 s, unsmoothed (MAE-DFER track)
PATIENT_GLOB = "*.mp4"

# ---------- comparison (step 6) ----------
MAEDFER_CSV = None       # e.g. DRIVE_DIR / "results/maedfer_test_predictions.csv" (layout: see compare_runs.py)

RESULTS.mkdir(parents=True, exist_ok=True)
for p, label in [(CKPT_MAFW, "MAFW checkpoint"), (PATIENT_VIDEOS, "patient videos")]:
    print(f"{label:16s} {'OK ' if Path(p).exists() else 'MISSING'}  {p}")

In [ ]:
# Unpack the face crops on Colab (once per session, onto the fast local disk)
for target, tar in TARS.items():
    if Path(target).exists():
        print("already unpacked:", target)
    elif Path(tar).exists():
        Path(target).parent.mkdir(parents=True, exist_ok=True)
        print("unpacking", tar, "...")
        sh(f"tar -xf '{tar}' -C '{Path(target).parent}'")
    else:
        print("not found (fine if you do not need it):", tar)

n_clips = len([d for d in ER_FRAMES.iterdir() if d.is_dir()]) if ER_FRAMES.exists() else 0
print(f"\nElderReact clips with face crops: {n_clips} (expected 1323)")
print("ElderReact label files:", sorted(p.name for p in ER_ANNO.glob("*.csv")) if ER_ANNO.exists() else "MISSING")

## Step 2 — Train / validation split
Pools the official train (615) and dev (355) clips and re-splits them into ~824 train / ~146 validation clips with a fixed seed — the same protocol as the MAE-DFER track. With `SPLIT_FILE` set, the validation clips are taken from that list instead (to use *exactly* the MAE-DFER split). The official **test** set (353 clips) is never touched.

In [ ]:
split_args = ["--split", SPLIT, "--val-frac", VAL_FRAC, "--split-seed", SPLIT_SEED]
if SPLIT_FILE:
    split_args += ["--split-file", SPLIT_FILE]
COMMON = ["--train", ER_ANNO / "train.csv", "--dev", ER_ANNO / "dev.csv", "--frames", ER_FRAMES]

if not done(SPLIT_DIR / "val_split.csv"):
    run("train_elderreact.py", "--init", CKPT_MAFW, *COMMON, "--out", SPLIT_DIR, *split_args, "--split-only")
VAL_CSV, TEST_CSV = SPLIT_DIR / "val_split.csv", ER_ANNO / "test.csv"
print("validation list:", SPLIT_DIR / "val_clips.txt")

## Step 3 — Baseline: the MAFW model on ElderReact before fine-tuning
The released S2D MAFW model (fold 1), no ElderReact training. Its scores for the 6 shared emotions are used directly. F1 thresholds are tuned on the validation set and then applied to the test set.

In [ ]:
BASE_VAL, BASE_TEST = RESULTS / f"baseline_{SPLIT}_val", RESULTS / "baseline_test"
if not done(BASE_VAL / "thresholds.json"):
    run("evaluate.py", "--mode", "er-zeroshot", "--sampling", "uniform", "--ckpt", CKPT_MAFW,
        "--anno", VAL_CSV, "--frames", ER_FRAMES, "--out", BASE_VAL)
if not done(BASE_TEST / "metrics.txt"):
    run("evaluate.py", "--mode", "er-zeroshot", "--sampling", "uniform", "--ckpt", CKPT_MAFW,
        "--anno", TEST_CSV, "--frames", ER_FRAMES, "--out", BASE_TEST, "--thresholds", BASE_VAL / "thresholds.json")
show_text(BASE_TEST / "metrics.txt")

## Step 4 — Fine-tune S2D on ElderReact
Starts from the MAFW model; the classifier is cut to the 6 ElderReact emotions. Only the landmark prompts (MCP), temporal adapters (TMA) and classifier are trained (or only the classifier with `TRAINABLE = "head"`). The best epoch (by validation loss) is saved as `best.pth`; training stops early when validation stops improving.

In [ ]:
RUN_DIR = RESULTS / RUN_NAME
if not done(RUN_DIR / "log.csv"):
    run("train_elderreact.py", "--init", CKPT_MAFW, *COMMON, "--out", RUN_DIR, *split_args,
        "--epochs", EPOCHS, "--patience", PATIENCE, "--select", SELECT, "--lr", LR, "--warmup", 1,
        "--trainable", TRAINABLE, "--batch", BATCH, "--accum", ACCUM)

In [ ]:
# Training curves
import csv
import matplotlib.pyplot as plt
rows = list(csv.DictReader(open(RUN_DIR / "log.csv")))
ep = [int(r["epoch"]) for r in rows]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
a1.plot(ep, [float(r["train_loss"]) for r in rows], label="train loss", color="#2a78d6")
a1.plot(ep, [float(r["val_loss"]) for r in rows], label="validation loss", color="#eb6834")
a1.set_xlabel("epoch"); a1.set_title("loss (lower = better)", loc="left"); a1.legend(frameon=False)
a2.plot(ep, [float(r["val_mean_auc"]) for r in rows], color="#1baf7a")
a2.set_xlabel("epoch"); a2.set_title("validation mean AUC (higher = better)", loc="left")
for a in (a1, a2):
    a.spines[["top", "right"]].set_visible(False); a.grid(axis="y", color="#e4e3dd")
best = torch.load(RUN_DIR / "best.pth", map_location="cpu", weights_only=False)
print(f"best epoch: {best['epoch']}  (validation loss {best['val_loss']:.4f}, mean AUC {100 * best['val_mean_auc']:.2f})")
plt.tight_layout(); plt.show()

## Step 5 — Evaluate the fine-tuned model (validation → thresholds → test)

In [ ]:
FT_VAL, FT_TEST = RESULTS / f"{RUN_NAME}_val", RESULTS / f"{RUN_NAME}_test"
FT_CKPT = RUN_DIR / "best.pth"
if not done(FT_VAL / "thresholds.json"):
    run("evaluate.py", "--mode", "er", "--sampling", "uniform", "--ckpt", FT_CKPT,
        "--anno", VAL_CSV, "--frames", ER_FRAMES, "--out", FT_VAL)
if not done(FT_TEST / "metrics.txt"):
    run("evaluate.py", "--mode", "er", "--sampling", "uniform", "--ckpt", FT_CKPT,
        "--anno", TEST_CSV, "--frames", ER_FRAMES, "--out", FT_TEST, "--thresholds", FT_VAL / "thresholds.json")
show_text(FT_TEST / "metrics.txt")

## Step 6 — Compare models on the test set (95 % bootstrap confidence intervals)
Reports AUC, mAP, F1 with validation-tuned thresholds and F1 at a fixed 0.5 cutoff (the MAE-DFER track's metrics), each with a confidence interval, plus the difference to the baseline. Add earlier runs or the MAE-DFER predictions (`MAEDFER_CSV` in the settings) to `COMPARE`.

In [ ]:
COMPARE = {"S2D before (MAFW)": BASE_TEST, f"S2D fine-tuned ({RUN_NAME})": FT_TEST}
# earlier runs, if they exist (official split, from the terminal):
for name, folder in [("S2D 10 ep, official split", RESULTS / "er_ft_short_test")]:
    if (folder / "predictions.csv").exists():
        COMPARE[name] = folder
if MAEDFER_CSV and Path(MAEDFER_CSV).exists():
    COMPARE["MAE-DFER"] = MAEDFER_CSV

run("compare_runs.py", "--runs", *COMPARE.values(), "--names", *COMPARE.keys(),
    "--out", RESULTS / f"compare_{RUN_NAME}.txt")

## Step 7 — t-SNE: high-level features before vs after fine-tuning (ElderReact test)

In [ ]:
TSNE_PNG = RESULTS / f"tsne_{RUN_NAME}.png"
if not done(TSNE_PNG):
    run("plot_tsne.py", "--inputs", BASE_TEST / "features.npz", FT_TEST / "features.npz",
        "--titles", "Before (MAFW model)", "After ElderReact fine-tuning", "--out", TSNE_PNG)
display(Image(str(TSNE_PNG), width=900))

## Step 8 — 3 annotated frames per patient video
Moments at 25 %, 50 % and 75 % of each video; face box + emotion scores. Made with both models: the fine-tuned one (6 emotions, no "neutral") and the MAFW one (11 emotions incl. "neutral").

In [ ]:
videos = sorted(PATIENT_VIDEOS.glob(PATIENT_GLOB))
print(len(videos), "videos:", [v.name for v in videos])
if not videos:
    raise SystemExit(f"No videos matching {PATIENT_GLOB} in {PATIENT_VIDEOS}")
for mode, ckpt, sub in [("er", FT_CKPT, f"annotated_{RUN_NAME}_{WINDOW_PRESET}"),
                        ("mafw", CKPT_MAFW, f"annotated_mafw_{WINDOW_PRESET}")]:
    out = RESULTS / sub
    if not done(out / f"{videos[-1].stem}_3frames.png"):
        run("annotate_video.py", "--mode", mode, "--ckpt", ckpt, "--preset", WINDOW_PRESET,
            "--video", *videos, "--out", out)
for v in videos:
    display(Markdown(f"**{v.name}**"))
    display(Image(str(RESULTS / f"annotated_{RUN_NAME}_{WINDOW_PRESET}" / f"{v.stem}_3frames.png"), width=700))

## Step 9 — Emotion timeline per patient video
Windows over the whole video (preset `WINDOW_PRESET`), both models. The coloured band shows the dominant emotion; `*_segments.csv` lists the stretches where it stays the same (= where the expression changes).

In [ ]:
TL_DIR = RESULTS / f"timeline_{RUN_NAME}_{WINDOW_PRESET}"
if not done(TL_DIR / f"{videos[-1].stem}_timeline.png"):
    run("timeline_video.py", "--video", *videos, "--er-ckpt", FT_CKPT, "--mafw-ckpt", CKPT_MAFW,
        "--preset", WINDOW_PRESET, "--out", TL_DIR, *(["--tmp-dir", "/content"] if IN_COLAB else []))
for v in videos:
    display(Markdown(f"**{v.name}**"))
    display(Image(str(TL_DIR / f"{v.stem}_timeline.png"), width=1000))

In [ ]:
# Stretches with the same dominant emotion (first video; change the index for the others)
import csv
seg = list(csv.DictReader(open(TL_DIR / f"{videos[0].stem}_segments.csv")))
print(f"{'model':6s} {'emotion':18s} {'from':>6s} {'to':>6s} {'seconds':>8s}")
for r in seg:
    print(f"{r['model']:6s} {r['dominant_emotion']:18s} {r['start']:>6s} {r['end']:>6s} {float(r['duration_s']):8.1f}")

## Optional A — face-crop new videos
Only needed for videos that do not have crops yet (the ElderReact and MAFW crops come from the tar files). Set `RUN_CROP = True` and the folders below.

In [ ]:
RUN_CROP = False
CROP_VIDEOS = [REPO / "data/ElderReact_Data/ElderReact_test"]     # folder(s) with videos
CROP_OUT = DATA / "ElderReact/faces_224"
if RUN_CROP:
    run("preprocess_faces.py", "--videos", *CROP_VIDEOS, "--out", CROP_OUT)

## Optional B — MAFW fold-1 check of the released model
Reproduces our MAFW number (48.18 WAR / 33.59 UAR with uniform sampling). Needs `MAFW_test_deepface.tar` (Colab) or the crops in `data/processed/MAFW/test_deepface` (local).

In [ ]:
RUN_MAFW = False
if RUN_MAFW and not done(RESULTS / "mafw_fold1_uniform/metrics.txt"):
    run("evaluate.py", "--ckpt", CKPT_MAFW, "--anno", MAFW_ANNO, "--frames", MAFW_FRAMES,
        "--out", RESULTS / "mafw_fold1_uniform", "--sampling", "uniform")
if (RESULTS / "mafw_fold1_uniform/metrics.txt").exists():
    show_text(RESULTS / "mafw_fold1_uniform/metrics.txt")

## Troubleshooting
* **CUDA out of memory** → lower `BATCH` in the settings (and raise `ACCUM` so `BATCH × ACCUM` stays 16).
* **Colab disconnected** → run all cells again from the top; finished steps are skipped (results are on Drive). Training restarts from the beginning, though.
* **"MISSING" in the settings check** → compare your Drive folders with the layout under *Before you start*.
* **A step should be redone** → set `REDO = True`, or delete that step's folder in `results/`.
* **Different split** → change `SPLIT` / `SPLIT_FILE`; a new `RUN_NAME` keeps the old results.